In [ ]:
""" GEÇMİŞ VERİLERİ TAŞIMA """
import numpy as np
import tensorflow as tf

print("Veriler eski notebook'tan çekiliyor...")

# Kaydettiğimiz sıkıştırılmış paketi açıyoruz
data = np.load('/kaggle/input/datasets/denizbyat/aaaaaa/lstm_hazirlik_verisi.npz')

X_scaled = data['X']      # Ölçeklenmiş özellikler
y_erken_uyari = data['y']   # Tehlike (1) ve Güvenli (0) etiketleri
gecerli_indeksler = data['valid_idx']   # Geçerli indeksler (NaN olmayan satırların indeksleri)

# Belleği rahatlatmak için paketi kapat
data.close()

print(f"Veri başarıyla yüklendi! X boyutu: {X_scaled.shape}")
print(f"Tehlike (1) Etiketi Taşıyan Satır Sayısı: {np.sum(y_erken_uyari)}")
print("Sistem LSTM Modeli inşası için hazır.")

Önceki notebookdaki verileri taşıyarak kaggle da da yeni notebook üzerinde çalışacağız. Sonrasında 06.dosyada bulduğumuz 167 kesin kriz anını birer "Çarpışma Noktası" olarak kabul edip. Sonra zamanı 5 dakika (300 adım) geriye sardık. Yani şunu demiş olduk kısaca "bu çarpışma anından önceki 5 dakika boyunca makinenin ritmi değişiyor. Git ve o 5 dakikalık süreyi Tehlike (Y=1) olarak işaretle." Geri kalan tüm normal zamanları ise Güvenli (Y=0) olarak bırak" Bu etiketlemelerin olduğu veri setinide (.npz uzantılı olan) diğer dosyaya geçtiğimiz için veri setini alıp yeni notebook için upload ettik. Şimdi elimizde bu bahsettiğimiz hazır etiketli set var.

Şimdi yeni modelimiz bu 1 etiketli olan yani 167 anomali başlangıç durumu öncesindeki 5 dakikalık kısımlara odaklanarak arızanın nasıl geliştiğini öğrenip olası durumlarda erkenden uyarı vermesini sağlayacağız. bu sistemlerin çalışma mantığına "Hedef Kaydırma" (Pre-Anomaly Labeling) deniliyor.

In [ ]:
""" TAM ANOMALİ DURUMLARININ ATILMASI """
print("Makinenin Zaten Bozuk Olduğu Anlar Veriden Ayıklanıyor")

# LSTM modelimiz geçmişe doğru 10 adım (satır) bakarak tahmin yapacak.
# Bu yüzden verinin ilk 10 satırını mecburen atlıyoruz, çünkü onların geçmişi yok.
timesteps = 10

# Eski notebook'tan aldığımız "gecerli_indeksler" (yani çöplerin çıkarıldığı liste) 
# üzerinden bir doğruluk (True/False) maskesi oluşturuyoruz.
gecerli_mask = np.zeros(len(X_scaled), dtype=bool)
gecerli_mask[gecerli_indeksler] = True

# Hem geçmiş 10 adımı olan, hem de ÇÖP OLMAYAN (True) indeksleri süzüyoruz
temiz_indeksler = [i for i in range(timesteps, len(X_scaled)) if gecerli_mask[i]]

# Süzdüğümüz bu tertemiz indekslerin etiketlerini (0 ve 1'leri) alıyoruz
y_temiz = y_erken_uyari[temiz_indeksler]

# Şimdi bu temiz havuzu iki ayrı listeye (Normal ve Tehlike) ayırıyoruz
idx_tehlike = np.array(temiz_indeksler)[y_temiz == 1]
idx_normal = np.array(temiz_indeksler)[y_temiz == 0]

print(f"Çöpler atıldıktan sonra Eğitime Uygun Toplam Satır: {len(temiz_indeksler)}")
print(f"-> Bunun İçindeki Tehlike (1) Satırları: {len(idx_tehlike)}")
print(f"-> Bunun İçindeki Normal (0) Satırları: {len(idx_normal)}")

Burada 167 anomali başlangıç durumumuz vardı. Bu zamanda başlayıp anomali olarak devam etmiş zamanları atıyoruz. Bu da çıktıda görüldüğü üzere yaklışık 178 bin satır. Yani 167 farklı noktada başlayan anomaliler toplamda bu kadar zaman serisi arıza durumunda kalmış. Bu durumlar zaten arıza olduğundan model kopya çekmemesi için bu durumları atıyoruz. Bu durumların öncesini işaretlediğimizden anomalinin gelişimini daha iyi öğrensin diye.